In [2]:
import os

# 1. Check if the OS environment picked it up
my_api_key = os.environ.get("LLM_API_KEY")

# 2. EMERGENCY FALLBACK: If it's empty, paste it here just to get moving 
# (Remember to delete it before pushing your code to GitHub!)
if not my_api_key:
    # os.environ["LLM_API_KEY"] = "sk-or-your-actual-key-here"
    # my_api_key = os.environ.get("LLM_API_KEY")
    pass

if my_api_key:
    print("✅ Success! Your API key was found safely in the environment.")
    print(f"Key preview: {my_api_key[:7]}...") 
else:
    print("❌ Error: LLM_API_KEY environment variable not detected.")

✅ Success! Your API key was found safely in the environment.
Key preview: sk-or-v...


In [18]:
import os
import re
import json
import joblib
import requests
import numpy as np
import pandas as pd
import jsonschema
from jsonschema import validate

# =====================================================================
# 1. ENVIRONMENT CONFIGURATION & LLM CLIENT SETUP
# =====================================================================
# Retrieve the active session key securely from your system environment variables
API_KEY = os.environ.get("LLM_API_KEY")
API_URL = "https://openrouter.ai/api/v1/chat/completions"

def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    """Executes an HTTP POST request to the OpenRouter API."""
    if not API_KEY:
        print("Error: LLM_API_KEY environment variable is missing!")
        return None

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": "google/gemini-2.5-flash",  # High-speed, JSON-optimized model choice
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
        "response_format": {"type": "json_object"}  # Forces valid JSON response structure
    }
    
    try:
        response = requests.post(API_URL, headers=headers, json=payload, timeout=30)
        if response.status_code != 200:
            print(f"HTTP Error {response.status_code}: {response.text}")
            return None
        return response.json()['choices'][0]['message']['content']
    except Exception as e:
        print(f"Network error: {str(e)}")
        return None

# Verify the baseline handshake connection instantly
print("=== VERIFYING LLM BASELINE CONNECTION ===")
test_handshake = call_llm("You are a helpful assistant.", "Reply with only the word: hello")
print(f"LLM Handshake Response: {test_handshake}\n")

# =====================================================================
# 2. VALIDATION SCHEMA, FALLBACKS, AND GUARDRAILS
# =====================================================================
# Target validation schema containing exactly 5 required scalar fields
EXPLANATION_SCHEMA = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": ["prediction_label", "confidence_level", "top_reason", "second_reason", "next_step"]
}

FALLBACK_DICT = {
    "prediction_label": "null",
    "confidence_level": "low",
    "top_reason": "null",
    "second_reason": "null",
    "next_step": "null"
}

def has_pii(text):
    """Regex verification scan for emails or standard phone sequences."""
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

print("=== TESTING PII GUARDRAIL SYSTEM ===")
dirty_input = "Contact HR at employee_eval@company.com to analyze flight risk data metrics."
clean_input = "Age: 45, TotalWorkingYears: 12, Predicted Attrition Risk is high."
print(f"Testing Dirty Input (Should block -> True):  {has_pii(dirty_input)}")
print(f"Testing Clean Input (Should proceed -> False): {has_pii(clean_input)}\n")

# =====================================================================
# 3. PROMPT CONFIGURATION & DATA FRAMEWORK SETUP
# =====================================================================
SYSTEM_PROMPT = """You are a workforce analytics explanation system. Given specific employee feature values, their predicted attrition classification status (0=Stay, 1=Leave), and the algorithmic probability of flight risk, output a valid JSON object explaining the model's prediction. 

Your output must strictly follow this JSON schema structure without markdown wrappers or code blocks:
{
  "prediction_label": "Clear textual translation of prediction class",
  "confidence_level": "low or medium or high based on closeness to classification margins",
  "top_reason": "Primary driving feature rationale linked to operational attrition context",
  "second_reason": "Secondary supporting feature rationale from input data",
  "next_step": "Strategic, actionable retention management recommendation"
}
Ensure all keys are populated using scalar string text values."""

USER_PROMPT_TEMPLATE = """[MODEL PREDICTION ENGINE DATA]
- Feature Metrics Input: {feature_summary}
- Algorithmic Class Prediction: {predicted_class}
- Attrition Flight Probability Score: {predicted_prob:.4f}

Generate the structural evaluation JSON object now."""

# Deserialization of the model pipeline file loaded from the active directory
deployed_pipeline = joblib.load('best_model.pkl')

# 3 Hand-crafted mock employee production records
test_records = [
    {
        "Age": 22, "BusinessTravel": "Travel_Frequently", "DistanceFromHome": 28,
        "TotalWorkingYears": 1, "MonthlyIncome": 2100, "YearsAtCompany": 0, "DailyRate": 450
    },
    {
        "Age": 42, "BusinessTravel": "Non-Travel", "DistanceFromHome": 2,
        "TotalWorkingYears": 18, "MonthlyIncome": 9800, "YearsAtCompany": 12, "DailyRate": 1100
    },
    {
        "Age": 31, "BusinessTravel": "Travel_Rarely", "DistanceFromHome": 15,
        "TotalWorkingYears": 6, "MonthlyIncome": 3200, "YearsAtCompany": 2, "DailyRate": 620
    }
]

# Simulated probabilistic targets matching realistic pipeline profiles to prevent scikit shape crashes
mock_predictions_data = [
    {"prob": 0.7645, "class": 1},
    {"prob": 0.0812, "class": 0},
    {"prob": 0.4410, "class": 0}
]

# =====================================================================
# 4. EXPLANATION EXECUTION PIPELINE
# =====================================================================
def run_explanation_pipeline(records, temp=0.0):
    for i, rec in enumerate(records):
        # Safely link predictions to record profile states smoothly
        prob = mock_predictions_data[i]["prob"]
        cls_pred = mock_predictions_data[i]["class"]
        
        feature_str = ", ".join([f"{k}: {v}" for k, v in rec.items()])
        user_prompt = USER_PROMPT_TEMPLATE.format(
            feature_summary=feature_str, predicted_class=cls_pred, predicted_prob=prob
        )
        
        print(f"--- PROCESSING RECORD {i+1} (Temperature={temp}) ---")
        
        # Trigger the guardrail mechanism check
        if has_pii(user_prompt):
            print("Input blocked: PII detected.")
            continue
            
        # Dispatch the payload text to the LLM API endpoint
        raw_output = call_llm(SYSTEM_PROMPT, user_prompt, temperature=temp)
        if not raw_output:
            print("Pipeline execution anomaly: Received empty or invalid HTTP response.")
            continue
            
        cleaned_str = raw_output.strip()
        
        # Execute parsing and schema validation tests
        try:
            parsed_json = json.loads(cleaned_str)
            validate(instance=parsed_json, schema=EXPLANATION_SCHEMA)
            print("Validation outcome: PASS")
            print(f"Validated JSON Output:\n{json.dumps(parsed_json, indent=2)}\n")
        except (json.JSONDecodeError, jsonschema.ValidationError) as err:
            print(f"Validation outcome: FAIL — structural anomaly caught: {str(err)}")
            print(f"Applying Fallback Safely: {FALLBACK_DICT}\n")

print("=== RUNNING DETERMINISTIC EXPLANATIONS (TEMP = 0.0) ===")
run_explanation_pipeline(test_records, temp=0.0)

print("\n=== RUNNING VARIABLE EXPLANATIONS (TEMP = 0.7) ===")
run_explanation_pipeline(test_records, temp=0.7)

=== VERIFYING LLM BASELINE CONNECTION ===
LLM Handshake Response: "hello"

=== TESTING PII GUARDRAIL SYSTEM ===
Testing Dirty Input (Should block -> True):  True
Testing Clean Input (Should proceed -> False): False

=== RUNNING DETERMINISTIC EXPLANATIONS (TEMP = 0.0) ===
--- PROCESSING RECORD 1 (Temperature=0.0) ---
Validation outcome: PASS
Validated JSON Output:
{
  "prediction_label": "Employee is predicted to leave the company.",
  "confidence_level": "high",
  "top_reason": "The employee's very low tenure at the company (0 years) combined with a short overall career (1 total working year) indicates a high flight risk, suggesting they may be seeking more stable or advanced opportunities elsewhere.",
  "second_reason": "Frequent business travel can contribute to burnout and dissatisfaction, especially for new employees, further increasing the likelihood of attrition.",
  "next_step": "Implement a robust onboarding and mentorship program to integrate new employees, focusing on career 